# étoiles de type Herbig
$\rightarrow$ **analyse de v1296Aql**

## la cible 
- V1295 Aql est une étoile de Herbig Ae (type A) de masse intermédiaire (plusieurs fois la masse du Soleil).
- Sa forte gravité requiert des vitesses orbitales très élevées à proximité de l'étoile.

## les données 
|||
|---|---|
|OBJECT|	V1295Aql|
|EXPTIME2|	5 x 300 s|
|SPE_RPOW|	653|
|BSS_VHEL|	0|
|DATE-OBS|	2025-08-22T20:19:20.609|
|BSS_SITE|	RENNES|
|BSS_INST| SW400/FD5 + Dados200 + 25mic + ATIK420M|
|||






## spectro dashboard
- lancer la cellule suivante
- sur 'Colormap', bouton droit : "create new view for cell output"
- redimensionner ou déplacer l'onglet créé 'Output View' 

In [1]:
%matplotlib widget
import numpy as np
from spectro_dashboard import SpectroDashboard

# 1. Afficher le dashboard
db = SpectroDashboard()
db.show()


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.modeling import models, fitting
from astropy import units as u
from specutils import Spectrum, SpectralRegion
from specutils.manipulation import extract_region
from specutils.analysis import centroid, fwhm, equivalent_width, snr_derived
from astropy.stats import mad_std
from astropy.coordinates import SkyCoord, EarthLocation
from astropy.time import Time



from PIL import Image
db.show_image(Image.open('data/plouis/v1295.jpg').convert('L'), 'brut')


# chargement
filename = 'data/plouis/_v1295aql_20250822_847.fits'

# Lecture
sp = Spectrum.read(filename)
x_data = sp.spectral_axis.value
y_data = sp.flux.value 

# on affiche le tout
db.show_spectrum(x_data, y_data, label='Spectre Brut')


--> Affichage de l'image brut : bin=1, shape=(1418, 2148), min=0, avg=82.4, max=232, std=16.4
--> Affichage du spectre Spectre Brut : dispersion=0.8755 Å/px


# ajustement et mesure de H beta

In [3]:
# on relève la vitesse barycentrique de la cible
# Coordonnées de la cible
coord = SkyCoord.from_name("v1295 aql")

# Localisation du CALC
location = EarthLocation.from_geodetic(lon=-1.68*u.deg, lat=48.11*u.deg, height=50*u.m)

# Date d'observation depuis le header FITS
date_obs = Time(sp.meta['header']['DATE-OBS'], format="isot", scale="utc")
#print(f"{date_obs=}")

# Correction barycentrique
corr = coord.radial_velocity_correction(obstime=date_obs, location=location).to(u.km/u.s)
#print(f"Correction barycentrique : {corr:.2f}")

# Extraction des valeurs brutes (pour éviter les soucis avec les unités astropy)
x_data = sp.spectral_axis.value
y_data = sp.flux.value 

# Découpage de la zone (4910 * u.AA, 4940 * u.AA)
mask = (x_data > 4840) & (x_data < 4880)
x_window = x_data[mask]
y_window = y_data[mask]

# --- MESURES SANS AJUSTEMENT ---
c_kms = 299792.458 * u.km / u.s
lambda_0 = 4861.32 # H-beta repos
fwhm_neon_pix = 5   #px

region_win = 10
region_raie_V = SpectralRegion((4855.0 - region_win) * u.AA, (4855.0 + region_win )* u.AA)
region_raie_R = SpectralRegion((4868.0 - region_win) * u.AA, (4868.0 + region_win )* u.AA)

cent_raie_V = centroid(sp, regions=region_raie_V) 
fwhm_raie_V = fwhm(sp, regions=region_raie_V) 

cent_raie_R = centroid(sp, regions=region_raie_R) 
fwhm_raie_R = fwhm(sp, regions=region_raie_R) 

V_R = abs(cent_raie_V - cent_raie_R)
V_R_centre =  abs((cent_raie_V + cent_raie_R) / 2.0)
v_radiale = c_kms * ((V_R_centre.value / lambda_0) - 1)
v_rot_disk = (c_kms * (V_R / lambda_0)) / 2.0

# Incertitudes Statistiques
delta_lambda = np.mean(np.diff(sp.spectral_axis)) # Dispersion (A/pix)

region_cont = SpectralRegion(5000.0 * u.AA, 5300.0 * u.AA) 
sub_spectrum = extract_region(sp, region_cont)
flux_zone = sub_spectrum.flux.value
snr_val = np.median(flux_zone) / mad_std(flux_zone)

print (f"{delta_lambda=:.4f}, {snr_val=:.0f}")

err_cent_raie = fwhm_raie_V.value / (2.35 * snr_val) # on suppose les deux raies égales en FWHM

err_vr_stat = (c_kms * (err_cent_raie / lambda_0))
fwhm_inst_angstrom = fwhm_neon_pix * delta_lambda
sigma_syst_rv = (fwhm_inst_angstrom / 10.0 / lambda_0) * c_kms # Règle du 1/10eme

err_vr_total = np.sqrt(err_vr_stat.value**2 + sigma_syst_rv.value**2) # Ajout quadratique


print("-" * 40)
print(f"Position du Pic Bleu  : {cent_raie_V.value:.3f} +/- {err_cent_raie:.03f} A")
print(f"Position du Pic Rouge : {cent_raie_R.value:.3f} +/- {err_cent_raie:.03f} A")
print(f"Séparation            : {V_R.value:.3f} A")
print(f"Centre                           : {V_R_centre.value:.3f} A")
print(f"Vitesse radiale centre           : {v_radiale.value:.3f} km/s ({V_R_centre.value - lambda_0:.3f} A)")
print(f"Vitesse radiale centre (corrigée): {(v_radiale.value + corr.value):.3f} km/s")
print(f"VITESSE DE ROTATION DU DISQUE    : {v_rot_disk.value:.1f} +/- {err_vr_total:.1f} km/s")
print("-" * 40)


# on affiche le tout
db.clear_spectra()
db.show_spectrum(x_window, y_window, label='brut')

# Les Lignes Verticales (Centres des Gaussiennes)
l_v = db.ax_spec.axvline(cent_raie_V.value, color='blue', linestyle='-', alpha=0.8, label=f'Centre V')
l_r = db.ax_spec.axvline(cent_raie_R.value, color='green', linestyle='-', alpha=0.8, label=f'Centre R')



delta_lambda=0.8755 Angstrom, snr_val=26
----------------------------------------
Position du Pic Bleu  : 4854.905 +/- 0.312 A
Position du Pic Rouge : 4868.009 +/- 0.312 A
Séparation            : 13.104 A
Centre                           : 4861.457 A
Vitesse radiale centre           : 8.445 km/s (0.137 A)
Vitesse radiale centre (corrigée): -3.048 km/s
VITESSE DE ROTATION DU DISQUE    : 404.0 +/- 33.1 km/s
----------------------------------------
--> Affichage du spectre brut : dispersion=0.8755 Å/px


## analyse :

Cette vitesse représente la vitesse de rotation du gaz à l'endroit où la raie $\text{H}\beta$ est émise (son rayon de formation), projetée sur la ligne de visée (v sin(i))

- Une vitesse de plus $400 \text{ km/s}$ est une indication que le $\text{H}\beta$ est formé dans la région la plus interne et la plus chaude du disque d'accrétion, là où le gaz est proche de la vitesse limite de rotation stable autour d'une étoile massive.


# ajustement et mesure de Fe I

In [4]:
db.clear_spectra()

# Extraction des valeurs brutes (pour éviter les soucis avec les unités astropy)
x_data = sp.spectral_axis.value
y_data = sp.flux.value 


def mesure_raie (lam):
    lambda_0 = lam
    # Découpage de la zone (4910 * u.AA, 4940 * u.AA) $4923.
    mask = (x_data > (lambda_0 - 10)) & ((x_data < lambda_0 + 10))
    #mask = (x_data > 4917) & (x_data < 4934)
    x_window = x_data[mask]
    y_window = y_data[mask]
    
    # Initialisation pour aider les modèles
    y_max = np.max(y_window)
    y_min = np.min(y_window)
    
    # modele sur la raie 
    g_init= models.Gaussian1D(amplitude=y_max, mean=lambda_0, stddev=2.0)
    
    # on prépare le continuum 
    continuum_init = models.Const1D(amplitude=y_min)
    
    model_init = g_init + continuum_init
    
    # On ajuste sur les données fenêtrées
    fitter = fitting.LevMarLSQFitter()
    fit = fitter(model_init, x_window, y_window)
    
    # fit[0] est la Gaussienne V, fit[1] est la Gaussienne R
    center_V = fit[0].mean.value
    
    # Calcul de la vitesse
    delta_lambda = abs(lambda_0 - center_V)
    
    c = 299792.458
    v_rot_disk = (c * (delta_lambda / lambda_0))

    print("-" * 40)
    print(f"Position du Pic (ajusté)  : {center_V:.3f} A")
    print(f"Delta Lambda              : {delta_lambda:.3f} A")
    print(f"Vitesse radiale (corrigée): {(v_rot_disk + corr.value):.1f} km/s")
    print(f"Vitesse radiale (/ centre): {(v_rot_disk) - (v_radiale.value):.1f} km/s")
    
    # on affiche le tout
    #db.show_spectrum(x_window, y_window, label='Spectre Brut')
    #db.show_spectrum(x_window, fit(x_window), label='Fit Gaussien')
    
    # Les Lignes Verticales (Centres des Gaussiennes)
    #l_v = db.ax_spec.axvline(center_V, color='blue', linestyle='-', alpha=0.8, label=f'Centre')

print(f"VITESSE BARYCENTRALE OBSERVATEUR : {corr.value:.1f} km/s")

for lam in (4923.93, # FeII repos
            5018.44,  # FeII repos
            5169.03) : # FeII repos
    mesure_raie (lam)
    


# et les résultats
print("-" * 40)
print(f"Vitesse radiale centre (corrigée): {(v_radiale.value + corr.value):.3f} km/s")
print("-" * 40)

print (f"ATTENTION : incertitudes importantes (SNR = {snr_val:.0f}) -> +/- {err_vr_total:.2f} km/s")

VITESSE BARYCENTRALE OBSERVATEUR : -11.5 km/s
----------------------------------------
Position du Pic (ajusté)  : 4924.551 A
Delta Lambda              : 0.621 A
Vitesse radiale (corrigée): 26.3 km/s
Vitesse radiale (/ centre): 29.3 km/s
----------------------------------------
Position du Pic (ajusté)  : 5018.549 A
Delta Lambda              : 0.109 A
Vitesse radiale (corrigée): -5.0 km/s
Vitesse radiale (/ centre): -2.0 km/s
----------------------------------------
Position du Pic (ajusté)  : 5169.790 A
Delta Lambda              : 0.760 A
Vitesse radiale (corrigée): 32.6 km/s
Vitesse radiale (/ centre): 35.7 km/s
----------------------------------------
Vitesse radiale centre (corrigée): -3.048 km/s
----------------------------------------
ATTENTION : incertitudes importantes (SNR = 26) -> +/- 33.14 km/s


# analyse

Résumé :

- On a une structure gazeuse autour de cette étoile de Herbig Ae qui comprend :

|Raie|Vitesse / Centre|Interprétation|
|:----|:----|:----|
|Fe II 5018|−2.0 km/s|Le gaz est presque à la même vitesse que l'étoile|
|Fe II 4924|+29.3 km/s|Le gaz tombe sur l'étoile -> accrétion|
|Fe II 5169|+35.7 km/s|-> accrétion plus rapide|

-> **Une jeune étoile en construction**

Pour rappel : 


| Profil Classique	| Cause Physique	| Signature Spectrale | 
| ---| ---| ---| 
| P-Cygni	| Vent/Éjection (Expansion)	| Composante d'Absorption Bleue + Composante d'Émission Rouge| 
| P-Cygni Inversé| 	Accrétion (Contraction)	| Composante d'Émission Bleue + Composante d'Absorption Rouge| 
|  H$\beta$ V1295 Aql	| Hybride (Accrétion + Éjection + Rotation) | Absorption générale + Double Inversion| 

